# Fine-tuning con Transformers

En este notebook se implementará un modelo basado en transformers para clasificar los reportes mamográficos en categorías BI-RADS.  
Como referencia, se utilizará el mejor baseline clásico obtenido en el notebook anterior: LinearSVC con balanceo de clases.

El objetivo de esta etapa es comparar el desempeño de un modelo profundo frente a los modelos clásicos de TF-IDF.

In [2]:
# Librerías básicas
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Carga del dataset limpio
df = pd.read_csv("../data/processed/reports_cleaned.csv")

# Selección de variables
df_model = df[["Full_Report_clean", "BI-RADS"]].copy()
df_model.columns = ["text", "label"]

# Limpieza mínima
df_model = df_model.dropna(subset=["text", "label"])
df_model["text"] = df_model["text"].astype(str)
df_model["label"] = df_model["label"].astype(int)

# Split estratificado
train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model["label"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nDistribución train:")
print(train_df["label"].value_counts().sort_index())
print("\nDistribución test:")
print(test_df["label"].value_counts().sort_index())

Train shape: (3485, 2)
Test shape: (872, 2)

Distribución train:
label
0     773
1     477
2    2108
3      69
4      41
5      13
6       4
Name: count, dtype: int64

Distribución test:
label
0    193
1    119
2    527
3     18
4     11
5      3
6      1
Name: count, dtype: int64


## Preparación para fine-tuning

En esta sección se preparan los datos para el entrenamiento del modelo transformer.  
Primero se convierten los conjuntos de entrenamiento y prueba al formato requerido por la librería `datasets`, y luego se aplica la tokenización del texto.

Como modelo inicial se utilizará DistilBERT, debido a que ofrece un buen equilibrio entre rendimiento y costo computacional.  
Este modelo ya fue preentrenado en grandes corpus de texto y puede ajustarse a tareas de clasificación supervisada con un número limitado de ejemplos.

In [3]:
# Si hace falta instalar dependencias, ejecuta esto una sola vez:
# !pip install transformers datasets accelerate evaluate

from datasets import Dataset
from transformers import AutoTokenizer

# Convertimos a formato Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Modelo base
model_name = "distilbert-base-uncased"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Función de tokenización
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# Aplicamos tokenización
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Revisamos una muestra
tokenized_train[0]

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3485 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

{'text': 'mamografia digitalizada bilateral craneo-caudal y medio lateral oblicuas y tecnica de eklund mamas con protesis de ubicacion retroglandular. las mamas se presentan heterogeneamente densas que podria ocultar algunos nodulos pequenos (acr tipo c). los implantes se presentan integros, sin deformaciones ni desplazamientos. no se observan calcificaciones sospechosas. imagenes nodulares con caracteres ganglionares, en la region pectoaxilar derecha. conclusion y valoracion: - mamas con protesis. - mamas con areas densas. ecografia mamaria anterior informa formacion quistica retroareolar derecha. - bi-rads 2 (segun la acr). recomendaciones: - se sugiere correlacion con ecografia mamaria actualizada y control mamografico segun criterio del medico tratante.',
 'label': 2,
 'input_ids': [101,
  5003,
  5302,
  17643,
  22749,
  3617,
  21335,
  2850,
  17758,
  11308,
  2080,
  1011,
  6187,
  14066,
  2140,
  1061,
  19960,
  3695,
  11457,
  27885,
  10415,
  6692,
  2015,
  1061,
  8

## Definición de etiquetas

A continuación, se define el número total de clases presentes en el problema.  
Esto es necesario para inicializar correctamente la capa de clasificación del modelo.

In [4]:
# Número de clases
num_labels = df_model["label"].nunique()
print("Número de clases:", num_labels)

# Etiquetas únicas
label_list = sorted(df_model["label"].unique().tolist())
print("Clases:", label_list)

id2label = {i: str(i) for i in label_list}
label2id = {str(i): i for i in label_list}

id2label, label2id

Número de clases: 7
Clases: [0, 1, 2, 3, 4, 5, 6]


({0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6'},
 {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6})

## Definición de etiquetas

El problema se formuló como una tarea de clasificación multiclase con 7 categorías BI-RADS.  
Las clases observadas en el conjunto de datos son: 0, 1, 2, 3, 4, 5 y 6.

Para configurar correctamente el modelo, se definieron los diccionarios `id2label` y `label2id`, que permiten mapear entre identificadores numéricos y etiquetas de clase.  
Esta configuración garantiza que la salida del modelo sea consistente durante el entrenamiento y la evaluación.

## Carga del modelo

En esta sección se carga el modelo base preentrenado que será ajustado para la tarea de clasificación multiclase.  
Se utilizará `AutoModelForSequenceClassification`, configurando explícitamente el número de clases y el mapeo entre identificadores y etiquetas.

In [6]:
from transformers import AutoModelForSequenceClassification

# Cargamos el modelo preentrenado para clasificación
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## Definición de métricas de evaluación

Para evaluar el desempeño del modelo durante el entrenamiento, se utilizarán las métricas de accuracy, Macro F1 y Weighted F1.  
Estas métricas permiten medir tanto el rendimiento global del modelo como su comportamiento en un escenario con posible desbalance entre clases.

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    weighted_f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

## Configuración del entrenamiento

En esta sección se definen los parámetros principales del proceso de fine-tuning.  
Estos incluyen el número de épocas, el tamaño de batch, la frecuencia de evaluación y la estrategia para guardar el mejor modelo durante el entrenamiento.

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_dir="./logs_distilbert",
    logging_strategy="epoch",
    report_to="none"
)

training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
eval_use_gather_object=False,
fp16=False,
fp16

## Preparación del entrenador

En esta sección se define el `data collator` y se inicializa el `Trainer`, que se encargará de ejecutar el proceso de fine-tuning y evaluación del modelo.

In [9]:
from transformers import DataCollatorWithPadding, Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer

/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_12475/3674095275.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
trainer.train()

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.468200,0.208097,0.927752,0.467202,0.916420
2,0.211500,0.228401,0.946101,0.485265,0.933486
3,0.165200,0.229139,0.941514,0.483568,0.931024


/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1308, training_loss=0.2816686251112445, metrics={'train_runtime': 217.9969, 'train_samples_per_second': 47.959, 'train_steps_per_second': 6.0, 'total_flos': 692535072867840.0, 'train_loss': 0.2816686251112445, 'epoch': 3.0})

## Resultados del fine-tuning

Durante el entrenamiento, la pérdida de entrenamiento disminuyó de forma consistente en las tres épocas, mientras que la pérdida de validación se mantuvo relativamente estable.  
Esto sugiere que el modelo aprendió patrones útiles del conjunto de entrenamiento sin mostrar un sobreajuste severo.

En términos de desempeño, la accuracy alcanzó valores cercanos a 0.94, pero el Macro F1 se mantuvo mucho más bajo que el Weighted F1.  
Esto indica que, aunque el modelo funciona bien en términos globales, todavía presenta dificultades para clasificar de forma equilibrada todas las clases BI-RADS, especialmente las menos frecuentes.

El warning relacionado con `pin_memory` en MPS no afecta la validez del entrenamiento, ya que corresponde únicamente a una limitación de optimización en Apple Silicon.

In [11]:
trainer.evaluate()

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.22840069234371185,
 'eval_accuracy': 0.9461009174311926,
 'eval_macro_f1': 0.4852650015229973,
 'eval_weighted_f1': 0.9334859328847245,
 'eval_runtime': 5.0118,
 'eval_samples_per_second': 173.989,
 'eval_steps_per_second': 21.749,
 'epoch': 3.0}

## Evaluación final del modelo transformer

La evaluación sobre el conjunto de prueba arrojó un accuracy de 0.946 y un Weighted F1 de 0.933, lo que refleja un buen rendimiento global del modelo.  
Sin embargo, el Macro F1 fue considerablemente menor, con un valor cercano a 0.485, lo que indica dificultades para mantener un desempeño equilibrado entre todas las clases BI-RADS.

Este patrón sugiere que el modelo transformer aprendió con éxito los patrones más frecuentes del conjunto de datos, pero todavía presenta limitaciones para clasificar de manera uniforme las categorías minoritarias.  

## Balanceo del transformer con class weights

Para mejorar el desempeño en las clases menos representadas, se probará una versión del transformer con pesos de clase en la función de pérdida.  
Esta estrategia permite penalizar más los errores en clases minoritarias sin modificar la distribución original del conjunto de datos.

En esta etapa se comparará el transformer base contra una versión balanceada utilizando la misma partición de entrenamiento y prueba.

In [12]:
import torch
import numpy as np
from torch import nn
from transformers import Trainer

# Calcular pesos por clase a partir del train
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = torch.tensor(class_weights, dtype=torch.float)

class_weights

tensor([0.0035, 0.0057, 0.0013, 0.0392, 0.0660, 0.2081, 0.6763])

Los pesos de clase se calcularon de forma inversamente proporcional a la frecuencia de cada categoría.  
De esta manera, las clases menos frecuentes obtienen mayor importancia durante el cálculo de la pérdida.

In [13]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

La clase personalizada `WeightedLossTrainer` reemplaza la función de pérdida estándar por una versión ponderada.  
Esto permite que el entrenamiento favorezca un aprendizaje más equilibrado entre las clases BI-RADS.

In [14]:
weighted_trainer = WeightedLossTrainer(
    model=AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    ),
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_12475/3976635437.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


En esta configuración se reutilizan los mismos datos tokenizados, pero se modifica únicamente el criterio de optimización.  
Así, la comparación con el modelo base será directa y metodológicamente consistente.

In [15]:
weighted_trainer.train()

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.125600,0.850812,0.917431,0.400126,0.899973
2,0.748600,0.782341,0.913991,0.397972,0.899009
3,0.652400,0.754805,0.933486,0.438399,0.924197


/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1308, training_loss=0.8421967532656608, metrics={'train_runtime': 215.2537, 'train_samples_per_second': 48.571, 'train_steps_per_second': 6.077, 'total_flos': 692535072867840.0, 'train_loss': 0.8421967532656608, 'epoch': 3.0})

In [16]:
weighted_trainer.evaluate()

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7548053860664368,
 'eval_accuracy': 0.9334862385321101,
 'eval_macro_f1': 0.43839933204565995,
 'eval_weighted_f1': 0.9241972344552379,
 'eval_runtime': 5.103,
 'eval_samples_per_second': 170.88,
 'eval_steps_per_second': 21.36,
 'epoch': 3.0}

Aunque la accuracy es elevada, el Macro F1 sigue siendo bajo, lo que confirma que el desbalance de clases continúa afectando el desempeño del modelo.  
El Weighted F1 alto indica que el rendimiento está concentrado en las clases más frecuentes.  
Por ello, el criterio principal para comparar las variantes será el Macro F1.

## Prueba de aumento de datos para clases minoritarias

Como el balanceo por pesos no mejoró de forma suficiente el Macro F1, se probará una estrategia de aumento de datos en las clases minoritarias.  
Esta técnica busca generar más variedad en el entrenamiento sin modificar el conjunto de prueba.

## Estrategia seleccionada

Se aplicará aumento de texto únicamente sobre el conjunto de entrenamiento.  
La idea es enriquecer las clases menos representadas con nuevas versiones de sus ejemplos, en lugar de repetir exactamente las mismas muestras.

## Criterio de comparación

La comparación entre modelos se realizará usando Macro F1 como métrica principal, porque refleja mejor el desempeño en clases desbalanceadas.  
También se observarán accuracy y weighted F1 para tener una visión completa del comportamiento del modelo.

## Identificación de clases minoritarias

Antes de aplicar augmentación, se revisará la distribución de clases en el conjunto de entrenamiento.  
Esto permite detectar qué etiquetas tienen menor representación y deben recibir más ejemplos sintéticos.

In [18]:
import pandas as pd

train_counts = train_df["label"].value_counts().sort_index()
train_counts

label
0     773
1     477
2    2108
3      69
4      41
5      13
6       4
Name: count, dtype: int64

## Selección de clases a augmentar

Se definirán como minoritarias aquellas clases cuya frecuencia esté por debajo del promedio del conjunto de entrenamiento.  
Solo estas clases serán aumentadas para evitar introducir sesgos innecesarios en las clases mayoritarias.

In [19]:
mean_count = train_counts.mean()
minority_classes = train_counts[train_counts < mean_count].index.tolist()
minority_classes

[1, 3, 4, 5, 6]

## Función de augmentación

Se aplicará una transformación simple y controlada sobre los textos de las clases minoritarias.  
La idea es generar variantes del texto sin alterar su significado clínico principal.

In [20]:
import random
import re

def simple_augment_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## Construcción del conjunto aumentado

Para cada ejemplo perteneciente a una clase minoritaria, se generará una copia augmentada.  
Luego se combinarán los datos originales con los nuevos ejemplos sintéticos para formar un nuevo conjunto de entrenamiento.

In [21]:
augmented_rows = []

for _, row in train_df.iterrows():
    augmented_rows.append(row.to_dict())
    if row["label"] in minority_classes:
        new_row = row.copy()
        new_row["text"] = simple_augment_text(row["text"])
        augmented_rows.append(new_row.to_dict())

train_df_aug = pd.DataFrame(augmented_rows).reset_index(drop=True)
train_df_aug["label"].value_counts().sort_index()

label
0     773
1     954
2    2108
3     138
4      82
5      26
6       8
Name: count, dtype: int64

## Verificación de la nueva distribución

Se revisará la distribución final del conjunto aumentado para confirmar que las clases minoritarias quedaron mejor representadas.  
Esta versión será la que se utilice para tokenizar y reentrenar el transformer.

In [22]:
train_df_aug["label"].value_counts().sort_index()

label
0     773
1     954
2    2108
3     138
4      82
5      26
6       8
Name: count, dtype: int64

## Preparación para el reentrenamiento

Con el conjunto aumentado ya construido, se repetirá el proceso de tokenización y entrenamiento usando la misma arquitectura del transformer base.  
Así será posible comparar ambos modelos bajo condiciones controladas.

## Augmentación textual más realista

Se generarán varias versiones sintéticas por cada ejemplo minoritario.  
La augmentación se hará de forma conservadora para mantener el significado clínico del informe y evitar introducir ruido semántico.

## Estrategia seleccionada

Para cada texto de una clase minoritaria se crearán varias variantes con cambios leves de redacción, sin modificar la etiqueta original.  
Esto permite aumentar la diversidad del entrenamiento sin duplicar exactamente los mismos registros.

In [24]:
import random
import re

random.seed(42)

synonyms = {
    "normal": ["habitual", "sin alteraciones", "conservado"],
    "aumento": ["incremento", "elevación", "mayor volumen"],
    "disminución": ["reducción", "descenso", "menor"],
    "presencia": ["existencia", "hallazgo", "detección"],
    "ausencia": ["no evidencia", "no se observa", "carencia"],
    "lesión": ["hallazgo", "alteración", "lesión"],
    "masa": ["formación", "lesión nodular", "masa"],
    "bordes": ["contornos", "límites", "márgenes"],
    "compatible": ["sugestivo", "concordante", "compatible"],
    "sospechoso": ["sugestivo", "de alerta", "sospechoso"]
}

def augment_text_basic(text, n_variants=3):
    text = str(text)
    variants = []
    for _ in range(n_variants):
        new_text = text
        for word, options in synonyms.items():
            if re.search(rf"\b{word}\b", new_text, flags=re.IGNORECASE):
                replacement = random.choice(options)
                new_text = re.sub(rf"\b{word}\b", replacement, new_text, flags=re.IGNORECASE)
        new_text = re.sub(r"\s+", " ", new_text).strip()
        variants.append(new_text)
    return variants

## Verificación de la distribución final

Se revisará cuántos ejemplos quedaron por clase después de la augmentación.  
El objetivo es que las clases minoritarias aumenten sin desbalancear en exceso el conjunto de entrenamiento.

In [25]:
train_df_aug["label"].value_counts().sort_index()

label
0     773
1     954
2    2108
3     138
4      82
5      26
6       8
Name: count, dtype: int64

## Evaluación del conjunto aumentado

Se evaluará el desempeño del transformer utilizando el conjunto de entrenamiento aumentado.  
El objetivo es verificar si la nueva distribución de clases produce alguna mejora respecto al modelo base.

La comparación se hará con las mismas métricas utilizadas previamente, manteniendo el conjunto de prueba sin cambios.

## Objetivo de la comparación

Esta prueba permitirá comprobar si el aumento de datos generó una diferencia real en el rendimiento del modelo.  
Si no hay mejoras relevantes, se descartará esta estrategia como solución principal al desbalance.

## Re-tokenización del conjunto aumentado

El conjunto aumentado debe convertirse primero a `Dataset` antes de aplicar la tokenización por lotes.  
Esto evita errores de compatibilidad entre pandas y Hugging Face Datasets.

In [28]:
from datasets import Dataset

train_dataset_aug = Dataset.from_pandas(train_df_aug)
tokenized_train_aug = train_dataset_aug.map(tokenize_function, batched=True)

Map:   0%|          | 0/4089 [00:00<?, ? examples/s]

In [29]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length")

In [30]:
aug_trainer = Trainer(
    model=AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    ),
    args=training_args,
    train_dataset=tokenized_train_aug,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

aug_trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_12475/1796552916.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  aug_trainer = Trainer(
/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.558700,0.249703,0.925459,0.403893,0.907509
2,0.234300,0.217936,0.928899,0.504589,0.925837
3,0.179600,0.191922,0.942661,0.573804,0.941891


/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1536, training_loss=0.3241967012484868, metrics={'train_runtime': 252.1214, 'train_samples_per_second': 48.655, 'train_steps_per_second': 6.092, 'total_flos': 812561237577216.0, 'train_loss': 0.3241967012484868, 'epoch': 3.0})

In [31]:
aug_results = aug_trainer.evaluate()
aug_results

/Users/sebas/Library/CloudStorage/OneDrive-postgrado.uv.cl/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.19192223250865936,
 'eval_accuracy': 0.9426605504587156,
 'eval_macro_f1': 0.5738039863925406,
 'eval_weighted_f1': 0.9418912864400306,
 'eval_runtime': 4.9602,
 'eval_samples_per_second': 175.8,
 'eval_steps_per_second': 21.975,
 'epoch': 3.0}

## Interpretación final

El aumento de datos sí produjo una mejora real en el desempeño del modelo, especialmente en la capacidad de recuperar clases minoritarias.  
Por esta razón, esta versión es más adecuada que la anterior para representar el comportamiento del clasificador.

## Comparación de modelos

Se compararon tres configuraciones del transformer: el modelo base con fine-tuning, la versión con pesos de clase y la versión con augmentación.  
La métrica principal para la selección fue el Macro F1, debido al desbalance de clases presente en el problema.

| Modelo | Eval loss | Accuracy | Macro F1 | Weighted F1 | Observación |
|---|---:|---:|---:|---:|---|
| Baseline / fine-tuning base | 0.2284 | 0.9461 | 0.4853 | 0.9335 | Mejoró sobre el entrenamiento inicial y sirve como referencia principal. |
| Transformer con class weights | 0.7548 | 0.9335 | 0.4384 | 0.9242 | No mostró mejora; quedó por debajo del baseline. |
| Transformer con augmentación | 0.1919 | 0.9427 | 0.5738 | 0.9419 | Obtuvo el mejor Macro F1 y la menor pérdida de evaluación. |

## Conclusión

Al comparar las tres configuraciones entrenadas, la versión con augmentación fue la que obtuvo el mejor desempeño, especialmente en Macro F1, que es la métrica más adecuada para un problema desbalanceado.

La versión con class weights no superó al baseline y presentó un rendimiento inferior en Macro F1 y accuracy.  
En consecuencia, el transformer entrenado con el conjunto aumentado será el modelo utilizado para el análisis final.

## Comparación final entre notebooks

Al comparar el mejor modelo del notebook del transformer con el mejor modelo del notebook de TFDF/Linear SVC balanced, se observa que el Linear SVC balanced presenta un rendimiento superior en la métrica principal de evaluación.

Aunque el transformer con augmentación mostró mejoras importantes, el Linear SVC balanced alcanzó un mejor equilibrio entre precisión y recall, y mantuvo un Macro F1 más alto en la evaluación más confiable realizada con validación cruzada.

Por esta razón, el modelo seleccionado para la implementación final es Linear SVC con class_weight='balanced'.

## Modelo final seleccionado

El modelo final para implementar es Linear SVC con class_weight='balanced', ya que mostró el mejor rendimiento global y una mejor estabilidad en la evaluación adicional.